In [ ]:
"""
Sepsis Prediction — Inference Script
======================================
Use this to run the trained model on new patients.
Requires:
  - saved_models/lstm_v4_final.pt
  - saved_models/lstm_v4_scaler.pkl
  - A folder of new .psv patient files
"""

import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — edit these
# ─────────────────────────────────────────────────────────────────────────────
MODEL_PATH    = "saved_models/lstm_v4_final.pt"
SCALER_PATH   = "saved_models/lstm_v4_scaler.pkl"
HEADER_FOLDER = "../processed_training_Ay"  # to get feature column names
NEW_DATA_DIR  = "../new_patients"           # folder of new .psv files to predict
OUTPUT_CSV    = "results/predictions.csv"
THRESHOLD     = 0.35   # use the recall-constrained threshold from training
MAX_TIMESTEPS = 48
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─────────────────────────────────────────────────────────────────────────────
# 1. Load feature columns (must match training)
# ─────────────────────────────────────────────────────────────────────────────
all_columns = set()
for fname in os.listdir(HEADER_FOLDER):
    if fname.endswith(".psv"):
        with open(os.path.join(HEADER_FOLDER, fname), "r") as fh:
            header = fh.readline().strip()
            if header:
                all_columns.update(header.split("|"))

FEATURE_COLUMNS = sorted(c for c in all_columns if c != "SepsisLabel")
N_FEATURES = len(FEATURE_COLUMNS)
print(f"Feature count: {N_FEATURES}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Model definition (must match training)
# ─────────────────────────────────────────────────────────────────────────────
class SepsisLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size    = input_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            dropout       = dropout if num_layers > 1 else 0.0,
            bidirectional = True,
        )
        lstm_out_dim = hidden_dim * 2
        self.attention  = nn.Linear(lstm_out_dim, 1)
        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        lstm_out, _  = self.lstm(x)
        attn_scores  = self.attention(lstm_out)
        attn_weights = torch.softmax(attn_scores, dim=1)
        context      = (lstm_out * attn_weights).sum(dim=1)
        return self.classifier(context)

# ─────────────────────────────────────────────────────────────────────────────
# 3. Load model and scaler
# ─────────────────────────────────────────────────────────────────────────────
model = SepsisLSTM(input_dim=N_FEATURES).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print(f"Model loaded from {MODEL_PATH}")

scaler = joblib.load(SCALER_PATH)
print(f"Scaler loaded from {SCALER_PATH}")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Predict on new patients
# ─────────────────────────────────────────────────────────────────────────────
def predict_patient(psv_path):
    """
    Returns: patient_id, sepsis_probability, prediction (0 or 1)
    """
    df = pd.read_csv(psv_path, sep="|")
    df = df.reindex(columns=FEATURE_COLUMNS + ["SepsisLabel"])
    df[FEATURE_COLUMNS] = df[FEATURE_COLUMNS].fillna(0.0)

    features = df[FEATURE_COLUMNS].values.astype(np.float32)
    L = len(features)

    # Pad to MAX_TIMESTEPS (keep last N hours)
    seq = np.zeros((MAX_TIMESTEPS, N_FEATURES), dtype=np.float32)
    if L >= MAX_TIMESTEPS:
        seq = features[-MAX_TIMESTEPS:]
    else:
        seq[MAX_TIMESTEPS - L:] = features

    # Normalise using the saved scaler
    seq = scaler.transform(seq).astype(np.float32)

    # Predict
    x = torch.tensor(seq).unsqueeze(0).to(DEVICE)   # (1, T, F)
    with torch.no_grad():
        logit = model(x).squeeze()
        prob  = torch.sigmoid(logit).item()

    pred = int(prob > THRESHOLD)
    return prob, pred


results = []
if os.path.exists(NEW_DATA_DIR):
    files = sorted(f for f in os.listdir(NEW_DATA_DIR) if f.endswith(".psv"))
    print(f"\nRunning inference on {len(files)} patients...")

    for fname in files:
        path = os.path.join(NEW_DATA_DIR, fname)
        prob, pred = predict_patient(path)
        patient_id = fname.replace(".psv", "")
        results.append({
            "patient_id":        patient_id,
            "sepsis_probability": round(prob, 4),
            "prediction":         pred,
            "alert":              "⚠ SEPSIS RISK" if pred == 1 else "OK",
        })
        print(f"  {patient_id:20} | prob: {prob:.4f} | {results[-1]['alert']}")

    df_out = pd.DataFrame(results)
    os.makedirs("results", exist_ok=True)
    df_out.to_csv(OUTPUT_CSV, index=False)
    print(f"\nPredictions saved → {OUTPUT_CSV}")
    print(f"Sepsis alerts: {df_out['prediction'].sum()} / {len(df_out)}")
else:
    print(f"\n⚠ New data folder not found: {NEW_DATA_DIR}")
    print("  Update NEW_DATA_DIR in the CONFIG section above.")
    print("\n  To predict a single patient manually:")
    print("  prob, pred = predict_patient('path/to/patient.psv')")